# HighRes AmbiStore

```{device-card} highres-ambistore
```

| Property | Value |
| --- | --- |
| Model | AmbiStore |
| Transport | TCP, port 1000 |
| Environment | Ambient |
| Driver support | Work in progress; not hardware-verified |

```{warning}
AmbiStore support has not been verified against AmbiStore hardware. Run motion only in a controlled setup and report verified behavior.
```

## How it talks

The store exposes a line-oriented TCP service. Each command receives an acknowledgement, optional data, and one completion status. The driver validates the echoed command and command ID and closes the connection if a timeout or malformed response makes the stream unsafe to reuse.

## Physical setup

Connect the store to a dedicated Ethernet interface. Supply clean dry air above 80 psi before homing or moving pneumatic doors. The factory service is normally `192.168.127.60:1000` and also answers at `10.253.253.253:1000`. Give the host interface an address on the corresponding isolated subnet, without a default gateway.

The example inventory below represents one known plate physically present in stacker 1, slot 1. Change the rack count, spots, slot heights, and assigned plates to match the actual machine before running motion.

## Describe the physical inventory

Carrier spots are zero-based in PLR and map to one-based device slots. The site height is a physical safety constraint used when selecting a destination.

In [ ]:
from pylabrobot.high_res.sample_storage import AmbiStore
from pylabrobot.resources import Coordinate, Plate, PlateCarrier, PlateHolder, Well

site = PlateHolder(
  name="stacker_1_slot_1",
  size_x=127.76,
  size_y=85.48,
  size_z=25.0,
  pedestal_size_z=0,
)
rack = PlateCarrier(name="stacker_1", size_x=130, size_y=90, size_z=600)
rack.assign_child_resource(site, location=Coordinate.zero(), spot=0)
well = Well(name="A1", size_x=8, size_y=8, size_z=10)
well.location = Coordinate(10, 10, 2)
plate = Plate(
  name="plate_1",
  size_x=127.76,
  size_y=85.48,
  size_z=14,
  ordered_items={"A1": well},
)
site.assign_child_resource(plate)
racks = [rack]

## Connect

`setup()` connects, reads version and environmental information where applicable, and discovers the actual transfer nests. It does not home unless `home=True` is passed.

In [ ]:
store = AmbiStore(host="192.168.127.60", name="ambistore", racks=racks)
await store.setup()

## Read version information

Confirm that the expected device answered.

In [ ]:
version = await store.request_version()
print(version)

## Inspect transfer nests

Compare the live sensor report with `store.nests`. Assign a `Plate` resource to any physically occupied nest before transfer operations.

In [ ]:
nest_status = await store.request_nest_status()
print(nest_status)

## Fetch a plate

This example moves the known plate from stacker 1, slot 1 to the first transfer nest. The driver requires that nest's live sensor to report clear.

In [ ]:
plate = await store.fetch_plate_to_loading_tray("plate_1", tray_index=0)

## Transfer between nests

If the machine reports at least two nests, move the plate from the first to the second. Both live sensors and PLR bookkeeping are validated before motion.

In [ ]:
plate = await store.transfer_plate_between_nests(
  source_tray_index=0, destination_tray_index=1
)

## Return the plate to storage

Move the plate from the second nest back to the smallest free site that is tall enough for it.

In [ ]:
plate = await store.take_in_plate(tray_index=1, site="smallest")

## Scan a barcode

Barcode scans require every transfer nest to be physically clear. `EMPTY` means no readable barcode, not necessarily no plate.

In [ ]:
barcodes = await store.request_stacker_barcodes(1, slot=1)
print(barcodes)

## Open robot doors

Ensure the carousel and automation interface are clear. If firmware reports an error, the driver waits for every robot door to reach the final open state.

In [ ]:
await store.open_all_doors()

## Close robot doors

Close and reseal the robot-access doors.

In [ ]:
await store.close_all_doors()

## Verify or recover the parked state

Recovery refuses to move while the spatula sensor reports a plate. Otherwise it retracts and homes only when the store is not safely parked.

In [ ]:
if not await store.request_is_parked():
  recovered = await store.recover()
  if not recovered:
    raise RuntimeError("Store did not recover to a parked state")

## Disconnect

Close the TCP connection when the workflow is finished.

In [ ]:
await store.stop()

## Reference

See the [sample-storage overview](../index.md) and [event reference](../events.md) for the full API.